In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask

In [ ]:
# ==========================================================
# Directories
# ==========================================================
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_ZSCORE = DIR_DATA / "2_processed" / "4_ICEYE_Zscore"
DIR_MASK = DIR_DATA / "1_org" / "a_mask"
DIR_ICEYE_MASK = DIR_DATA / "1-5_intermediate" / "4_ICEYE_Zscore_mask"

FILEPATH_ICEYE_LIST = DIR_METADATA / "iceye-list-full.csv"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"

In [ ]:
def create_evaluation_mask(
    patch_path,
    output_path,
    mask_gdf=None,
):
    """
    Create an evaluation mask aligned to a SAR/Z-score patch.

    If mask_gdf is provided:
        Pixels inside the polygon are 1.
        Pixels outside the polygon are 0.

    If mask_gdf is None:
        The entire patch is 1.
    """

    # ---------------------------------------
    # Read patch metadata
    # ---------------------------------------
    with rasterio.open(patch_path) as src:

        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        height = src.height
        width = src.width

    # ---------------------------------------
    # Create mask
    # ---------------------------------------
    if mask_gdf is None:

        # Entire patch is the evaluation area
        mask = np.ones(
            (height, width),
            dtype=bool,
        )

    else:

        # Make sure polygon uses patch CRS
        mask_gdf = mask_gdf.to_crs(crs)

        # Rasterize polygon
        mask = geometry_mask(
            mask_gdf.geometry,
            out_shape=(height, width),
            transform=transform,
            invert=True,
        )

    # ---------------------------------------
    # Output profile
    # ---------------------------------------
    mask_profile = profile.copy()

    mask_profile.update(
        dtype="uint8",
        count=1,
        nodata=0,
    )

    # ---------------------------------------
    # Save mask
    # ---------------------------------------
    with rasterio.open(
        output_path,
        "w",
        **mask_profile,
    ) as dst:

        dst.write(
            mask.astype(np.uint8),
            1,
        )

    return mask

In [ ]:
# ==========================================================
# Read metadata
# ==========================================================

iceye_df = pd.read_csv(FILEPATH_ICEYE_LIST)

patch_df = pd.read_csv(FILEPATH_PATCH_CENTERS)

iceye_df = iceye_df[
    iceye_df["zscore_mask"] == True
].copy()

patch_df = patch_df[
    patch_df["usage"] == "test"
].copy()

print(f"{len(iceye_df)} images")
print(f"{len(patch_df)} patches")


# ==========================================================
# Cache polygons
# ==========================================================

mask_cache = {}


# ==========================================================
# Create evaluation masks
# ==========================================================

for _, iceye_row in iceye_df.iterrows():

    image_id = iceye_row["iceye_id"]
    directory = iceye_row["dirname"]

    print(f"\nProcessing image {image_id}")

    image_dir = DIR_ZSCORE / directory
    output_dir = DIR_ICEYE_MASK / directory

    output_dir.mkdir(
        exist_ok=True,
        parents=True,
    )

    for _, patch_row in patch_df.iterrows():

        patch_id = patch_row["id"]
        area = patch_row["elora_area_id"]

        # --------------------------------------------------
        # Original Z-score patch
        # --------------------------------------------------

        patch_path = (
            image_dir
            / f"z_{image_id}_{patch_id}.tif"
        )

        if not patch_path.exists():
            continue

        # --------------------------------------------------
        # Output mask
        # --------------------------------------------------

        output_path = (
            output_dir
            / f"z_{image_id}_{patch_id}_mask.tif"
        )

        # --------------------------------------------------
        # Determine evaluation area
        # --------------------------------------------------

        if patch_row["control"] == False:

            # No evidenced flood polygon.
            # Evaluate the entire patch.
            mask_gdf = None

        else:

            # --------------------------------------------------
            # Read evidence polygon only once
            # --------------------------------------------------

            if area not in mask_cache:

                shp = (
                    DIR_MASK
                    / area
                    / f"{area}.shp"
                )

                mask_cache[area] = gpd.read_file(shp)

            mask_gdf = mask_cache[area]

        # --------------------------------------------------
        # Create evaluation mask
        # --------------------------------------------------

        create_evaluation_mask(
            patch_path=patch_path,
            output_path=output_path,
            mask_gdf=mask_gdf,
        )

print("Done.")